In [1]:
import json
from pathlib import Path
import pandas as pd
from shapely.geometry import Polygon
from shapely.validation import make_valid

In [2]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [6]:
def to_polygon(points):
    poly = Polygon(points)
    if not poly.is_valid:
        poly = make_valid(poly)
    if poly.is_empty or poly.area == 0:
        return None
    return poly


def extract_polygons_from_json(data):
    polys = []
    for pts in data.get("polys", []):
        poly = to_polygon(pts)
        if poly is not None:
            polys.append(poly)
    return polys


def intersection_area(p1, p2):
    inter = p1.intersection(p2)
    return inter.area if not inter.is_empty else 0.0


def union_area(p1, p2):
    uni = p1.union(p2)
    return uni.area if not uni.is_empty else 0.0


def iou(p1, p2):
    inter = intersection_area(p1, p2)
    uni = union_area(p1, p2)
    return inter / uni if uni > 0 else 0.0


def gt_coverage(jap_poly, gt_poly):
    inter = intersection_area(jap_poly, gt_poly)
    return inter / gt_poly.area if gt_poly.area > 0 else 0.0


def pred_coverage(jap_poly, gt_poly):
    inter = intersection_area(jap_poly, gt_poly)
    return inter / jap_poly.area if jap_poly.area > 0 else 0.0


def dice(p1, p2):
    inter = intersection_area(p1, p2)
    denom = p1.area + p2.area
    return (2.0 * inter) / denom if denom > 0 else 0.0


def best_jap_match_per_gt(jap_polys, gt_polys):
    """
    For each GT polygon, keep only the single Japanese polygon
    with the highest IoU.
    """
    matches = []

    for gt_idx, gt_poly in enumerate(gt_polys):
        best_jap_idx = None
        best_iou = -1.0
        best_jap_poly = None

        for jap_idx, jap_poly in enumerate(jap_polys):
            score = iou(jap_poly, gt_poly)
            if score > best_iou:
                best_iou = score
                best_jap_idx = jap_idx
                best_jap_poly = jap_poly

        if best_jap_poly is None:
            continue

        matches.append({
            "gt_idx": gt_idx,
            "jap_idx": best_jap_idx,
            "iou": best_iou,
            "gt_coverage": gt_coverage(best_jap_poly, gt_poly),
            "pred_coverage": pred_coverage(best_jap_poly, gt_poly),
            "dice": dice(best_jap_poly, gt_poly),
        })

    return matches


def summarize(matches):
    if not matches:
        return {
            "num_gt_boxes": 0,
            "num_matched_gt": 0,
            "mean_iou": 0.0,
            "median_iou": 0.0,
            "iou@0.3": 0.0,
            "iou@0.5": 0.0,
            "iou@0.7": 0.0,
            "mean_gt_coverage": 0.0,
            "mean_pred_coverage": 0.0,
            "mean_dice": 0.0,
        }

    df = pd.DataFrame(matches)

    return {
        "num_matched_gt": len(df),
        "mean_iou": df["iou"].mean(),
        "median_iou": df["iou"].median(),
        "iou@0.3": (df["iou"] >= 0.3).mean(),
        "iou@0.5": (df["iou"] >= 0.5).mean(),
        "iou@0.7": (df["iou"] >= 0.7).mean(),
        "mean_gt_coverage": df["gt_coverage"].mean(),
        "mean_pred_coverage": df["pred_coverage"].mean(),
        "mean_dice": df["dice"].mean(),
    }


def evaluate_file(jap_json_path, gt_json_path):
    jap_data = load_json(jap_json_path)
    gt_data = load_json(gt_json_path)

    jap_polys = extract_polygons_from_json(jap_data)
    gt_polys = extract_polygons_from_json(gt_data)

    matches = best_jap_match_per_gt(jap_polys, gt_polys)
    summary = summarize(matches)

    return {
        "file_name": Path(jap_json_path).name,
        "num_jap_polys": len(jap_polys),
        "num_gt_polys": len(gt_polys),
        "num_matched_gt": summary["num_matched_gt"],
        "mean_iou": summary["mean_iou"],
        "median_iou": summary["median_iou"],
        "iou@0.3": summary["iou@0.3"],
        "iou@0.5": summary["iou@0.5"],
        "iou@0.7": summary["iou@0.7"],
        "mean_gt_coverage": summary["mean_gt_coverage"],
        "mean_pred_coverage": summary["mean_pred_coverage"],
        "mean_dice": summary["mean_dice"],
    }


def evaluate_dirs_to_csv(jap_dir, gt_dir, out_csv):
    jap_dir = Path(jap_dir)
    gt_dir = Path(gt_dir)

    rows = []

    for jap_file in sorted(jap_dir.glob("*.json")):
        gt_file = gt_dir / jap_file.name

        if not gt_file.exists():
            rows.append({
                "file_name": jap_file.name,
                "num_jap_polys": None,
                "num_gt_polys": None,
                "num_matched_gt": None,
                "mean_iou": None,
                "median_iou": None,
                "iou@0.3": None,
                "iou@0.5": None,
                "iou@0.7": None,
                "mean_gt_coverage": None,
                "mean_pred_coverage": None,
                "mean_dice": None,
                "error": "Missing GT file",
            })
            continue

        try:
            row = evaluate_file(jap_file, gt_file)
            row["error"] = None
            rows.append(row)
        except Exception as e:
            rows.append({
                "file_name": jap_file.name,
                "num_jap_polys": None,
                "num_gt_polys": None,
                "num_matched_gt": None,
                "mean_iou": None,
                "median_iou": None,
                "iou@0.3": None,
                "iou@0.5": None,
                "iou@0.7": None,
                "mean_gt_coverage": None,
                "mean_pred_coverage": None,
                "mean_dice": None,
                "error": str(e),
            })

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df

In [4]:
jap_dir = r"outputs\jap_ocr_otsu\json"
gt_dir = r"outputs\en_ocr_otsu\json"
out_csv = r"ocr_overlap_metrics.csv"

In [ ]:
df = evaluate_dirs_to_csv(jap_dir, gt_dir, out_csv)
print(df)

# Method 2

In [8]:
from collections import defaultdict
from shapely.ops import unary_union  # keep if you later want union-based grouped IoU


def overlap_pred_ratio(pred_poly, gt_poly):
    """
    Overlap ratio wrt prediction area:
    intersection(pred, gt) / area(pred)
    """
    inter = intersection_area(pred_poly, gt_poly)
    return inter / pred_poly.area if pred_poly.area > 0 else 0.0


def match_predictions_to_gt(jap_polys, gt_polys, overlap_thresh=0.8):
    """
    Prediction-centric matching:
    A prediction is matched (+1) if ANY GT overlaps >= overlap_thresh
    based on overlap wrt prediction area.
    """
    matched = []

    for pred_idx, pred_poly in enumerate(jap_polys):
        best_gt_idx = None
        best_overlap = -1.0
        best_iou = -1.0

        for gt_idx, gt_poly in enumerate(gt_polys):
            ov = overlap_pred_ratio(pred_poly, gt_poly)
            if ov > best_overlap:
                best_overlap = ov
                best_gt_idx = gt_idx
                best_iou = iou(pred_poly, gt_poly)

        if best_gt_idx is not None and best_overlap >= overlap_thresh:
            matched.append({
                "pred_idx": pred_idx,
                "gt_idx": best_gt_idx,
                "pred_overlap": best_overlap,
                "pair_iou": best_iou,
            })

    return matched


def grouped_iou_total_area(matched_preds, jap_polys, gt_polys):
    """
    For each GT, aggregate all matched predictions assigned to it and compute IoU
    using TOTAL areas (as requested):
      IoU = total_intersection / (total_pred_area + gt_area - total_intersection)
    """
    gt_to_pred_idxs = defaultdict(list)
    for m in matched_preds:
        gt_to_pred_idxs[m["gt_idx"]].append(m["pred_idx"])

    rows = []
    for gt_idx, pred_idxs in gt_to_pred_idxs.items():
        gt_poly = gt_polys[gt_idx]

        total_pred_area = 0.0
        total_intersection = 0.0
        for pidx in pred_idxs:
            p = jap_polys[pidx]
            total_pred_area += p.area
            total_intersection += intersection_area(p, gt_poly)

        denom = total_pred_area + gt_poly.area - total_intersection
        giou = total_intersection / denom if denom > 0 else 0.0

        rows.append({
            "gt_idx": gt_idx,
            "num_preds_in_gt": len(pred_idxs),
            "group_iou": giou,
        })

    return rows


def summarize_predcentric(jap_polys, gt_polys, matched_preds, grouped_rows):
    num_pred = len(jap_polys)
    num_matched_pred = len(matched_preds)
    matched_pred_rate = (num_matched_pred / num_pred) if num_pred > 0 else 0.0

    pair_iou_vals = [m["pair_iou"] for m in matched_preds]
    group_iou_vals = [r["group_iou"] for r in grouped_rows]

    return {
        "num_pred_boxes": num_pred,
        "num_gt_boxes": len(gt_polys),
        "num_matched_pred": num_matched_pred,
        "matched_pred_rate": matched_pred_rate,  # your new metric
        "mean_pair_iou": (pd.Series(pair_iou_vals).mean() if pair_iou_vals else 0.0),
        "median_pair_iou": (pd.Series(pair_iou_vals).median() if pair_iou_vals else 0.0),
        "mean_grouped_iou": (pd.Series(group_iou_vals).mean() if group_iou_vals else 0.0),
        "num_gt_with_matches": len(grouped_rows),
    }


def evaluate_file(jap_json_path, gt_json_path, overlap_thresh=0.8):
    jap_data = load_json(jap_json_path)
    gt_data = load_json(gt_json_path)

    jap_polys = extract_polygons_from_json(jap_data)
    gt_polys = extract_polygons_from_json(gt_data)

    matched_preds = match_predictions_to_gt(jap_polys, gt_polys, overlap_thresh=overlap_thresh)
    grouped_rows = grouped_iou_total_area(matched_preds, jap_polys, gt_polys)
    summary = summarize_predcentric(jap_polys, gt_polys, matched_preds, grouped_rows)

    return {
        "file_name": Path(jap_json_path).name,
        "num_jap_polys": summary["num_pred_boxes"],
        "num_gt_polys": summary["num_gt_boxes"],
        "num_matched_pred": summary["num_matched_pred"],
        "matched_pred_rate": summary["matched_pred_rate"],
        "mean_pair_iou": summary["mean_pair_iou"],
        "median_pair_iou": summary["median_pair_iou"],
        "mean_grouped_iou": summary["mean_grouped_iou"],
        "num_gt_with_matches": summary["num_gt_with_matches"],
    }


def evaluate_dirs_to_csv(jap_dir, gt_dir, out_csv, overlap_thresh=0.8):
    jap_dir = Path(jap_dir)
    gt_dir = Path(gt_dir)

    rows = []

    for jap_file in sorted(jap_dir.glob("*.json")):
        gt_file = gt_dir / jap_file.name

        if not gt_file.exists():
            rows.append({
                "file_name": jap_file.name,
                "num_jap_polys": None,
                "num_gt_polys": None,
                "num_matched_pred": None,
                "matched_pred_rate": None,
                "mean_pair_iou": None,
                "median_pair_iou": None,
                "mean_grouped_iou": None,
                "num_gt_with_matches": None,
                "error": "Missing GT file",
            })
            continue

        try:
            row = evaluate_file(jap_file, gt_file, overlap_thresh=overlap_thresh)
            row["error"] = None
            rows.append(row)
        except Exception as e:
            rows.append({
                "file_name": jap_file.name,
                "num_jap_polys": None,
                "num_gt_polys": None,
                "num_matched_pred": None,
                "matched_pred_rate": None,
                "mean_pair_iou": None,
                "median_pair_iou": None,
                "mean_grouped_iou": None,
                "num_gt_with_matches": None,
                "error": str(e),
            })

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df

In [11]:
df = evaluate_dirs_to_csv(jap_dir, gt_dir, out_csv, overlap_thresh=0.1)
print(df)

   file_name  num_jap_polys  num_gt_polys  num_matched_pred  \
0  pg_1.json              9             8                 3   
1  pg_2.json             16            12                 5   
2  pg_3.json             18            13                13   
3  pg_4.json             12             8                 5   
4  pg_5.json             13            12                 5   
5  pg_6.json             18            16                 4   

   matched_pred_rate  mean_pair_iou  median_pair_iou  mean_grouped_iou  \
0           0.333333       0.170544         0.174234          0.170544   
1           0.312500       0.287644         0.238313          0.350374   
2           0.722222       0.303824         0.318135          0.327419   
3           0.416667       0.153719         0.066186          0.191760   
4           0.384615       0.222215         0.137792          0.222215   
5           0.222222       0.160880         0.161916          0.160880   

   num_gt_with_matches error  
0       

In [12]:
from collections import defaultdict
from pathlib import Path
import pandas as pd

def overlap_pred_ratio(pred_poly, gt_poly):
    inter = intersection_area(pred_poly, gt_poly)
    return inter / pred_poly.area if pred_poly.area > 0 else 0.0

def match_predictions_to_gt_m2(jap_polys, gt_polys, overlap_thresh=0.8):
    matched = []

    for pred_idx, pred_poly in enumerate(jap_polys):
        best_gt_idx = None
        best_overlap = -1.0
        best_iou = -1.0

        for gt_idx, gt_poly in enumerate(gt_polys):
            ov = overlap_pred_ratio(pred_poly, gt_poly)
            if ov > best_overlap:
                best_overlap = ov
                best_gt_idx = gt_idx
                best_iou = iou(pred_poly, gt_poly)

        if best_gt_idx is not None and best_overlap >= overlap_thresh:
            matched.append({
                "pred_idx": pred_idx,
                "gt_idx": best_gt_idx,
                "pred_overlap": best_overlap,
                "pair_iou": best_iou,
            })

    return matched

def grouped_iou_total_area(matched_preds, jap_polys, gt_polys):
    gt_to_pred_idxs = defaultdict(list)
    for m in matched_preds:
        gt_to_pred_idxs[m["gt_idx"]].append(m["pred_idx"])

    rows = []
    for gt_idx, pred_idxs in gt_to_pred_idxs.items():
        gt_poly = gt_polys[gt_idx]

        total_pred_area = 0.0
        total_intersection = 0.0
        for pidx in pred_idxs:
            p = jap_polys[pidx]
            total_pred_area += p.area
            total_intersection += intersection_area(p, gt_poly)

        denom = total_pred_area + gt_poly.area - total_intersection
        giou = total_intersection / denom if denom > 0 else 0.0

        rows.append({
            "gt_idx": gt_idx,
            "num_preds_in_gt": len(pred_idxs),
            "group_iou": giou,
        })

    return rows

def summarize_predcentric(jap_polys, gt_polys, matched_preds, grouped_rows):
    num_pred = len(jap_polys)
    num_matched_pred = len(matched_preds)
    matched_pred_rate = (num_matched_pred / num_pred) if num_pred > 0 else 0.0

    pair_iou_vals = [m["pair_iou"] for m in matched_preds]
    group_iou_vals = [r["group_iou"] for r in grouped_rows]

    return {
        "num_pred_boxes": num_pred,
        "num_gt_boxes": len(gt_polys),
        "num_matched_pred": num_matched_pred,
        "matched_pred_rate": matched_pred_rate,
        "mean_pair_iou": (pd.Series(pair_iou_vals).mean() if pair_iou_vals else 0.0),
        "median_pair_iou": (pd.Series(pair_iou_vals).median() if pair_iou_vals else 0.0),
        "mean_grouped_iou": (pd.Series(group_iou_vals).mean() if group_iou_vals else 0.0),
        "num_gt_with_matches": len(grouped_rows),
    }

def evaluate_file_m2(jap_json_path, gt_json_path, overlap_thresh=0.8):
    jap_data = load_json(jap_json_path)
    gt_data = load_json(gt_json_path)

    jap_polys = extract_polygons_from_json(jap_data)
    gt_polys = extract_polygons_from_json(gt_data)

    matched_preds = match_predictions_to_gt_m2(jap_polys, gt_polys, overlap_thresh=overlap_thresh)
    grouped_rows = grouped_iou_total_area(matched_preds, jap_polys, gt_polys)
    summary = summarize_predcentric(jap_polys, gt_polys, matched_preds, grouped_rows)

    return {
        "file_name": Path(jap_json_path).name,
        "num_jap_polys": summary["num_pred_boxes"],
        "num_gt_polys": summary["num_gt_boxes"],
        "num_matched_pred": summary["num_matched_pred"],
        "matched_pred_rate": summary["matched_pred_rate"],
        "mean_pair_iou": summary["mean_pair_iou"],
        "median_pair_iou": summary["median_pair_iou"],
        "mean_grouped_iou": summary["mean_grouped_iou"],
        "num_gt_with_matches": summary["num_gt_with_matches"],
    }

def evaluate_dirs_to_csv_m2(jap_dir, gt_dir, out_csv, overlap_thresh=0.8):
    jap_dir = Path(jap_dir)
    gt_dir = Path(gt_dir)

    rows = []
    for jap_file in sorted(jap_dir.glob("*.json")):
        gt_file = gt_dir / jap_file.name
        if not gt_file.exists():
            rows.append({"file_name": jap_file.name, "error": "Missing GT file"})
            continue

        try:
            row = evaluate_file_m2(jap_file, gt_file, overlap_thresh=overlap_thresh)
            row["error"] = None
            rows.append(row)
        except Exception as e:
            rows.append({"file_name": jap_file.name, "error": str(e)})

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df

In [13]:
from collections import defaultdict
from pathlib import Path
import pandas as pd

def overlap_pred_ratio(pred_poly, gt_poly):
    inter = intersection_area(pred_poly, gt_poly)
    return inter / pred_poly.area if pred_poly.area > 0 else 0.0

def overlap_gt_ratio(pred_poly, gt_poly):
    inter = intersection_area(pred_poly, gt_poly)
    return inter / gt_poly.area if gt_poly.area > 0 else 0.0

def match_predictions_to_gt_m3(
    jap_polys,
    gt_polys,
    pred_thresh=0.8,
    gt_thresh=0.3,
    rule="or",   # "or" or "and"
):
    matched = []

    for pred_idx, pred_poly in enumerate(jap_polys):
        best = None
        best_score = -1.0

        for gt_idx, gt_poly in enumerate(gt_polys):
            p_cov = overlap_pred_ratio(pred_poly, gt_poly)
            g_cov = overlap_gt_ratio(pred_poly, gt_poly)
            score = max(p_cov, g_cov)  # choose strongest overlap signal

            if score > best_score:
                best_score = score
                best = {
                    "pred_idx": pred_idx,
                    "gt_idx": gt_idx,
                    "pred_overlap": p_cov,
                    "gt_overlap": g_cov,
                    "pair_iou": iou(pred_poly, gt_poly),
                }

        if best is None:
            continue

        pass_match = (
            (best["pred_overlap"] >= pred_thresh or best["gt_overlap"] >= gt_thresh)
            if rule == "or"
            else
            (best["pred_overlap"] >= pred_thresh and best["gt_overlap"] >= gt_thresh)
        )

        if pass_match:
            matched.append(best)

    return matched

def grouped_iou_total_area(matched_preds, jap_polys, gt_polys):
    gt_to_pred_idxs = defaultdict(list)
    for m in matched_preds:
        gt_to_pred_idxs[m["gt_idx"]].append(m["pred_idx"])

    rows = []
    for gt_idx, pred_idxs in gt_to_pred_idxs.items():
        gt_poly = gt_polys[gt_idx]

        total_pred_area = 0.0
        total_intersection = 0.0
        for pidx in pred_idxs:
            p = jap_polys[pidx]
            total_pred_area += p.area
            total_intersection += intersection_area(p, gt_poly)

        denom = total_pred_area + gt_poly.area - total_intersection
        giou = total_intersection / denom if denom > 0 else 0.0

        rows.append({
            "gt_idx": gt_idx,
            "num_preds_in_gt": len(pred_idxs),
            "group_iou": giou,
        })

    return rows

def summarize_predcentric_m3(jap_polys, gt_polys, matched_preds, grouped_rows):
    num_pred = len(jap_polys)
    num_matched_pred = len(matched_preds)
    matched_pred_rate = (num_matched_pred / num_pred) if num_pred > 0 else 0.0

    pair_iou_vals = [m["pair_iou"] for m in matched_preds]
    group_iou_vals = [r["group_iou"] for r in grouped_rows]

    return {
        "num_pred_boxes": num_pred,
        "num_gt_boxes": len(gt_polys),
        "num_matched_pred": num_matched_pred,
        "matched_pred_rate": matched_pred_rate,
        "mean_pred_overlap": (pd.Series([m["pred_overlap"] for m in matched_preds]).mean() if matched_preds else 0.0),
        "mean_gt_overlap": (pd.Series([m["gt_overlap"] for m in matched_preds]).mean() if matched_preds else 0.0),
        "mean_pair_iou": (pd.Series(pair_iou_vals).mean() if pair_iou_vals else 0.0),
        "median_pair_iou": (pd.Series(pair_iou_vals).median() if pair_iou_vals else 0.0),
        "mean_grouped_iou": (pd.Series(group_iou_vals).mean() if group_iou_vals else 0.0),
        "num_gt_with_matches": len(grouped_rows),
    }

def evaluate_file_m3(jap_json_path, gt_json_path, pred_thresh=0.8, gt_thresh=0.3, rule="or"):
    jap_data = load_json(jap_json_path)
    gt_data = load_json(gt_json_path)

    jap_polys = extract_polygons_from_json(jap_data)
    gt_polys = extract_polygons_from_json(gt_data)

    matched_preds = match_predictions_to_gt_m3(
        jap_polys,
        gt_polys,
        pred_thresh=pred_thresh,
        gt_thresh=gt_thresh,
        rule=rule,
    )
    grouped_rows = grouped_iou_total_area(matched_preds, jap_polys, gt_polys)
    summary = summarize_predcentric_m3(jap_polys, gt_polys, matched_preds, grouped_rows)

    return {
        "file_name": Path(jap_json_path).name,
        "num_jap_polys": summary["num_pred_boxes"],
        "num_gt_polys": summary["num_gt_boxes"],
        "num_matched_pred": summary["num_matched_pred"],
        "matched_pred_rate": summary["matched_pred_rate"],
        "mean_pred_overlap": summary["mean_pred_overlap"],
        "mean_gt_overlap": summary["mean_gt_overlap"],
        "mean_pair_iou": summary["mean_pair_iou"],
        "median_pair_iou": summary["median_pair_iou"],
        "mean_grouped_iou": summary["mean_grouped_iou"],
        "num_gt_with_matches": summary["num_gt_with_matches"],
    }

def evaluate_dirs_to_csv_m3(jap_dir, gt_dir, out_csv, pred_thresh=0.8, gt_thresh=0.3, rule="or"):
    jap_dir = Path(jap_dir)
    gt_dir = Path(gt_dir)

    rows = []
    for jap_file in sorted(jap_dir.glob("*.json")):
        gt_file = gt_dir / jap_file.name
        if not gt_file.exists():
            rows.append({"file_name": jap_file.name, "error": "Missing GT file"})
            continue

        try:
            row = evaluate_file_m3(
                jap_file,
                gt_file,
                pred_thresh=pred_thresh,
                gt_thresh=gt_thresh,
                rule=rule,
            )
            row["error"] = None
            rows.append(row)
        except Exception as e:
            rows.append({"file_name": jap_file.name, "error": str(e)})

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df

In [15]:
# Method 2
df_m2 = evaluate_dirs_to_csv_m2(jap_dir, gt_dir, "ocr_overlap_metrics_m2.csv", overlap_thresh=0.1)

# Method 3 (recommended start)
df_m3 = evaluate_dirs_to_csv_m3(
    jap_dir,
    gt_dir,
    "ocr_overlap_metrics_m3.csv",
    pred_thresh=0.5,
    gt_thresh=0.1,
    rule="or",
)

print(df_m2)
print(df_m3)

   file_name  num_jap_polys  num_gt_polys  num_matched_pred  \
0  pg_1.json              9             8                 3   
1  pg_2.json             16            12                 5   
2  pg_3.json             18            13                13   
3  pg_4.json             12             8                 5   
4  pg_5.json             13            12                 5   
5  pg_6.json             18            16                 4   

   matched_pred_rate  mean_pair_iou  median_pair_iou  mean_grouped_iou  \
0           0.333333       0.170544         0.174234          0.170544   
1           0.312500       0.287644         0.238313          0.350374   
2           0.722222       0.303824         0.318135          0.327419   
3           0.416667       0.153719         0.066186          0.191760   
4           0.384615       0.222215         0.137792          0.222215   
5           0.222222       0.160880         0.161916          0.160880   

   num_gt_with_matches error  
0       

In [17]:
from pathlib import Path
import json
import pandas as pd
from shapely.geometry import Polygon
from shapely.validation import make_valid
from shapely.ops import unary_union

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def to_polygon(points):
    p = Polygon(points)
    if not p.is_valid:
        p = make_valid(p)
    if p.is_empty or p.area <= 0:
        return None
    return p

def extract_polys_and_scores(data, score_min=0.0, min_area=20.0):
    polys = []
    scores = data.get("scores", [1.0] * len(data.get("polys", [])))
    for pts, sc in zip(data.get("polys", []), scores):
        if sc < score_min:
            continue
        p = to_polygon(pts)
        if p is None:
            continue
        if p.area < min_area:
            continue
        polys.append(p)
    return polys

def inter_area(a, b):
    x = a.intersection(b)
    return 0.0 if x.is_empty else x.area

def iou(a, b):
    u = a.union(b)
    if u.is_empty:
        return 0.0
    return inter_area(a, b) / u.area

def iom(a, b):
    den = min(a.area, b.area)
    return (inter_area(a, b) / den) if den > 0 else 0.0

def should_link_for_merge(a, b, iom_thr=0.15, iou_thr=0.05, dist_thr=45):
    if iom(a, b) >= iom_thr:
        return True
    if iou(a, b) >= iou_thr:
        return True
    if a.distance(b) <= dist_thr:
        return True
    return False

def connected_components(adj):
    n = len(adj)
    seen = [False] * n
    comps = []
    for i in range(n):
        if seen[i]:
            continue
        stack = [i]
        seen[i] = True
        comp = []
        while stack:
            u = stack.pop()
            comp.append(u)
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
        comps.append(comp)
    return comps

def merge_fragmented_polys(polys, iom_thr=0.15, iou_thr=0.05, dist_thr=45):
    n = len(polys)
    if n == 0:
        return []
    adj = [[] for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            if should_link_for_merge(polys[i], polys[j], iom_thr, iou_thr, dist_thr):
                adj[i].append(j)
                adj[j].append(i)

    comps = connected_components(adj)
    merged = []
    for comp in comps:
        group = [polys[k] for k in comp]
        merged.append(unary_union(group))
    return merged

def should_link_pred_gt(p, g, iom_thr=0.2, iou_thr=0.08, dist_thr=120):
    if iom(p, g) >= iom_thr:
        return True
    if iou(p, g) >= iou_thr:
        return True
    if p.distance(g) <= dist_thr:
        return True
    return False

def bipartite_components(pred_polys, gt_polys, iom_thr=0.2, iou_thr=0.08, dist_thr=120):
    np_, ng = len(pred_polys), len(gt_polys)
    total = np_ + ng
    adj = [[] for _ in range(total)]

    for i, p in enumerate(pred_polys):
        for j, g in enumerate(gt_polys):
            if should_link_pred_gt(p, g, iom_thr, iou_thr, dist_thr):
                u = i
                v = np_ + j
                adj[u].append(v)
                adj[v].append(u)

    comps = connected_components(adj)

    out = []
    for comp in comps:
        p_idx = [x for x in comp if x < np_]
        g_idx = [x - np_ for x in comp if x >= np_]
        out.append((p_idx, g_idx))
    return out

def evaluate_file_m4(jap_json_path, gt_json_path,
                     score_min_pred=0.35, score_min_gt=0.35,
                     min_area=20.0):
    pred_data = load_json(jap_json_path)
    gt_data = load_json(gt_json_path)

    pred_raw = extract_polys_and_scores(pred_data, score_min=score_min_pred, min_area=min_area)
    gt_raw = extract_polys_and_scores(gt_data, score_min=score_min_gt, min_area=min_area)

    pred_polys = merge_fragmented_polys(pred_raw, iom_thr=0.12, iou_thr=0.04, dist_thr=10)
    gt_polys = merge_fragmented_polys(gt_raw, iom_thr=0.12, iou_thr=0.04, dist_thr=10)

    comps = bipartite_components(pred_polys, gt_polys, iom_thr=0.20, iou_thr=0.08, dist_thr=20)

    matched_pred_ids = set()
    matched_gt_ids = set()
    group_ious = []

    for p_ids, g_ids in comps:
        if len(p_ids) == 0 or len(g_ids) == 0:
            continue
        for i in p_ids:
            matched_pred_ids.add(i)
        for j in g_ids:
            matched_gt_ids.add(j)

        p_union = unary_union([pred_polys[i] for i in p_ids])
        g_union = unary_union([gt_polys[j] for j in g_ids])

        den = p_union.union(g_union).area
        giou = (p_union.intersection(g_union).area / den) if den > 0 else 0.0
        group_ious.append(giou)

    n_pred = len(pred_polys)
    n_gt = len(gt_polys)
    pred_rate = (len(matched_pred_ids) / n_pred) if n_pred > 0 else 0.0
    gt_rate = (len(matched_gt_ids) / n_gt) if n_gt > 0 else 0.0
    f1 = (2 * pred_rate * gt_rate / (pred_rate + gt_rate)) if (pred_rate + gt_rate) > 0 else 0.0

    return {
        "file_name": Path(jap_json_path).name,
        "num_pred_regions": n_pred,
        "num_gt_regions": n_gt,
        "matched_pred_rate": pred_rate,
        "matched_gt_rate": gt_rate,
        "f1_region_match": f1,
        "mean_group_iou": (pd.Series(group_ious).mean() if group_ious else 0.0),
        "median_group_iou": (pd.Series(group_ious).median() if group_ious else 0.0),
    }

def evaluate_dirs_to_csv_m4(jap_dir, gt_dir, out_csv):
    jap_dir = Path(jap_dir)
    gt_dir = Path(gt_dir)

    rows = []
    for jf in sorted(jap_dir.glob("*.json")):
        gf = gt_dir / jf.name
        if not gf.exists():
            rows.append({"file_name": jf.name, "error": "Missing GT file"})
            continue
        try:
            row = evaluate_file_m4(jf, gf)
            row["error"] = None
            rows.append(row)
        except Exception as e:
            rows.append({"file_name": jf.name, "error": str(e)})

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df

# Run
jap_dir = r"outputs\jap_ocr_otsu\json"
gt_dir = r"outputs\en_ocr_otsu\json"
df_m4 = evaluate_dirs_to_csv_m4(jap_dir, gt_dir, "ocr_overlap_metrics_m4.csv")
print(df_m4)

   file_name  num_pred_regions  num_gt_regions  matched_pred_rate  \
0  pg_1.json                 8               6           0.750000   
1  pg_2.json                12              12           0.666667   
2  pg_3.json                10              11           0.900000   
3  pg_4.json                 7               7           1.000000   
4  pg_5.json                13              11           0.615385   
5  pg_6.json                15              13           0.600000   

   matched_gt_rate  f1_region_match  mean_group_iou  median_group_iou error  
0         0.833333         0.789474        0.107631          0.102627  None  
1         0.666667         0.666667        0.180951          0.060298  None  
2         1.000000         0.947368        0.384697          0.327522  None  
3         1.000000         1.000000        0.052186          0.000207  None  
4         0.727273         0.666667        0.210699          0.111095  None  
5         0.615385         0.607595        0.090